# Day 23: Bridge to LLMs

Today we look **inside ChatGPT-style models**, layer by layer.

We use **one sentence** all day:

> **"ali drinks hot ___"** → the model must guess **chai**

**How today works:** we start with the simplest idea. Then we find where it **breaks**. The next layer fixes it. So every layer has a reason.

## What is an LLM?

**LLM = Large Language Model.**

| Word | Meaning |
|---|---|
| **Language** | It works with human words (English, Urdu, code). |
| **Model** | A program that learned patterns from examples. Your Day 19–22 networks were models too. |
| **Large** | It has a huge number of learned numbers inside. These are called **parameters** (or **weights**). |

**An LLM is a very big autocomplete.** It reads your text, guesses the next word, adds it, and guesses again. That is how it writes a full answer.

Examples: ChatGPT, Claude, Gemini, Llama.

**How big?**

| Model | Parameters (learned numbers) |
|---|---|
| Our Day 19 network | 147 |
| Our Day 20 network | 5,130 |
| Small language model (**SLM**) | 1 to 8 billion |
| Large language model (**LLM**) | 100 billion or more |

**SLM = Small Language Model.** Same idea, smaller size. It runs on a laptop or phone and is good at narrow jobs. *Example: an LLM is a big hospital with every kind of doctor. An SLM is a village clinic: fast and cheap, but not for everything.*

**Transformer** is the *design* (shape) of the network inside an LLM, like a CNN is the design for images. By the end of today you will have built a tiny one.

## The map for today

![Transformer, annotated](Visual%20Explanation/Day_23_Transformer_Annotated.png)

Look at it from the **bottom to the top**. We build one box at a time. At the end we come back to this picture and it will make sense.

| Layer | Idea | Problem it solves |
|---|---|---|
| 0 | The goal: autocomplete | (the goal) |
| 1 | Count and guess | – |
| 2 | Look at more words | Layer 1 gets a **tie** |
| 3 | Words become **numbers** | Layer 2 only knows words it has **seen** |
| 4 | **Attention** | A tie again, now with numbers |
| 5 | **Softmax** | Scores are not chances yet |
| 6 | **Repeat** | We only have one word |
| 7 | The real LLM | Our toy is tiny |

In [ ]:
import numpy as np                      # vectors and tables of numbers
from collections import Counter         # counts how many times each item appears
np.set_printoptions(precision=3, suppress=True)   # print 3 decimals, no e-notation


def softmax(scores):                    # same softmax as Day 21
    e = np.exp(scores)                  # 1. e to the power of each score
    return e / e.sum()                  # 2. divide by the total → chances that add up to 1

---
# Class Problem: "ali drinks hot ___"

## Layer 0: The goal is autocomplete

- Your phone keyboard suggests the next word. An LLM does the same, but much better.
- Its **only job**: look at the text so far, and guess the **next word**.
- Then it adds that word and guesses again.

```
ali drinks hot  →  chai
ali drinks hot chai  →  .
```

That is all. Every layer today is a better way to make this one guess.

**✏️ Quick Check:** Your phone keyboard: you type *"good"*. Which word will it suggest? Why does it know?

**Answer:** *morning* (or *luck*). The phone has seen *"good morning"* many times, so it guesses the word that came next most often. **We will do exactly this in Layer 1.**

## Layer 1: Count and guess

**Idea:** look at lots of text. After the word "hot", which word came **most often**? Guess that one.

*Everyday example:* a shopkeeper notices that after "hot", customers ask for chai most of the time. So he starts the chai.

Our "lots of text" is 4 small sentences:

In [ ]:
text = ["ali drinks hot chai",
        "sara drinks hot chai",
        "bilal eats hot roti",
        "ayesha eats hot roti"]

after_hot = Counter()                    # a counter: word → how many times
for sentence in text:
    w = sentence.split()                 # cut into words
    for i in range(len(w) - 1):          # look at each word and the one after it
        if w[i] == "hot":
            after_hot[w[i + 1]] += 1     # count the word that came after "hot"

print(after_hot)                         # chai 2, roti 2

**It is a tie** (chai 2, roti 2). Looking only at "hot", the model cannot decide.

**What breaks:** one word is not enough. We need to look at **more words**. → Layer 2

**✏️ Quick Check:** Count the words that come after **"drinks"**. Change `"hot"` in the code above.

In [ ]:
# your code here

**Answer:**

In [ ]:
after_drinks = Counter()
for sentence in text:
    w = sentence.split()
    for i in range(len(w) - 1):
        if w[i] == "drinks":
            after_drinks[w[i + 1]] += 1
print(after_drinks)    # hot 2: no tie here, "drinks" is always followed by "hot"

## Layer 2: Look at more words

**Idea:** do not use only the last word. Use the **last two words**.

- `drinks hot` → chai
- `eats hot` → roti

*Everyday example:* "hot" alone is not enough. "drinks hot" and "eats hot" are different questions. The word before it gives the clue.

In [ ]:
table = {}                                    # (word before, last word) → counter of next words
for sentence in text:
    w = sentence.split()
    for i in range(1, len(w) - 1):
        key = (w[i - 1], w[i])
        table.setdefault(key, Counter())[w[i + 1]] += 1

print(table[("drinks", "hot")])               # chai 2  → no tie now!
print(table[("eats", "hot")])                 # roti 2

The tie is gone. Looking at more words fixed it.

**What breaks:** try a phrase the table has **never seen**:

In [ ]:
print(table.get(("drinks", "warm")))          # None: the table has no answer

- The table only knows **exact phrases it has seen**. "warm" is almost the same as "hot", but the table does not know that.
- To remember every phrase, the table would grow without end.

**What we need:** a way to say *"warm" and "hot" mean almost the same thing*. For that, **words must become numbers**. → Layer 3

**✏️ Quick Check:** What does `table` say for the phrase `("ali", "drinks")`?

In [ ]:
# your code here

**Answer:** after *ali drinks* comes *hot*.

In [ ]:
print(table[("ali", "drinks")])    # hot 1

## Layer 3: Words become numbers

A computer cannot read words. We do **3 small steps**:

1. **Tokens:** cut the text into pieces.
2. **Token IDs:** give each piece a number (like a roll number).
3. **Embeddings:** give each piece a **list of numbers that describes its meaning**.

Then similar words get similar numbers, and the "warm = almost hot" problem goes away.

### 3a: Tokens

- Our toy: **1 word = 1 token** (cut at the spaces).

In [ ]:
sentence = "ali drinks hot"
tokens = sentence.split()        # cut at the spaces
print(tokens)                    # ['ali', 'drinks', 'hot']

**A real tokenizer** (the one used by GPT-4) cuts words into **pieces**.
If this cell gives an error, run `pip install tiktoken` once (it needs internet the first time).

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")          # the GPT-4 tokenizer
for text_ in ["Ali drinks hot chai.", "chaiwala", "unbelievable", "Lahore"]:
    ids_ = enc.encode(text_)                         # text → token IDs
    pieces = [enc.decode([i]) for i in ids_]         # each ID → its piece of text
    print(f"{text_!r:24} → {pieces}  ({len(ids_)} tokens)")

**✏️ Quick Check:** Use `enc.encode` to count the tokens in **"Sara"** and in **"Karachi"**.

In [ ]:
# your code here

**Answer:**

In [ ]:
for word in ["Sara", "Karachi"]:
    ids_ = enc.encode(word)
    print(word, "→", [enc.decode([i]) for i in ids_], len(ids_), "tokens")

### 3b: Token IDs

- The **vocabulary** is the list of all tokens the model knows.
- The **ID** of a token = its position in that list.
- Our vocabulary has 6 words. A real one has about 100,000.

In [ ]:
vocab = ["ali", "drinks", "eats", "hot", "chai", "roti"]   # position 0, 1, 2, 3, 4, 5

ids = [vocab.index(t) for t in tokens]    # find each token's position
print(ids)                                # [0, 1, 3]

**✏️ Quick Check:** What are the IDs of **"ali eats hot"**? Guess first, then check with code.

In [ ]:
# your code here

**Answer:**

In [ ]:
print([vocab.index(t) for t in "ali eats hot".split()])   # [0, 2, 3]

**Think:** what happens with `vocab.index("sara")`? Try it. Real tokenizers never get this error. Why? *(They cut unknown words into smaller pieces they do know, like "chaiwala" → chai + w + ala.)*

### 3c: Embeddings (vectors)

- An **ID is only a label.** ID 4 is not "bigger" or "closer" to ID 3 in meaning.
- So each token gets a **vector**: a list of numbers that says what the word is about.
- Our vectors have 4 numbers with names: **person, drink, food, warm**.
- Real models: 768 or more numbers per token, **learned** during training, with no names. We wrote ours so we can read them.

In [ ]:
#                person drink food warm
E = np.array([[3,     0,    0,   0],    # 0 ali
              [0,     2,    0,   0],    # 1 drinks
              [0,     0,    2,   0],    # 2 eats
              [0,     1,    1,   1],    # 3 hot   (drink? food? not sure!)
              [0,     2,    0,   1],    # 4 chai
              [0,     0,    2,   1]],   # 5 roti
             dtype=float)

vectors = E[ids]           # take the rows for our IDs [0, 1, 3]
print(vectors)

**✏️ Quick Check:** Print the vector of **roti** using its ID.

In [ ]:
# your code here

**Answer:**

In [ ]:
print(E[vocab.index("roti")])     # [0, 0, 2, 1]: food, warm

### 3d: Dot product = similarity

How do we measure if two words are close in meaning?

- **Multiply the matching numbers, then add.** This is the **dot product**.
- Big number = similar meaning. 0 = nothing in common.
- Same "multiply and add" as a CNN filter and a Dense neuron.

In [ ]:
chai = E[vocab.index("chai")]
for word in vocab:
    print(f"chai · {word:7} = {np.dot(chai, E[vocab.index(word)])}")

**✏️ Quick Check:** Which word is **most similar to roti** (not counting roti itself)? Use `np.dot`.

In [ ]:
# your code here

**Answer:**

In [ ]:
roti = E[vocab.index("roti")]
for word in vocab:
    print(f"roti · {word:7} = {np.dot(roti, E[vocab.index(word)])}")
# eats: 4  → roti is most similar to "eats"

**Link to Layer 2:** because meaning is now numbers, a word like "warm" (with a warm number) would get a vector close to "hot". The model no longer needs to have seen the exact phrase.

**What breaks:** look at the last word "hot" alone, now with numbers. → Layer 4

## Layer 4: Attention

### The problem (the same tie as Layer 1)

Use only the last word, "hot", to score every word in the vocabulary:

In [ ]:
hot = E[vocab.index("hot")]
for word, score in zip(vocab, E @ hot):     # E @ hot = dot product of hot with every row
    print(f"{word:7} {score}")
# chai 3, roti 3, hot 3 → a tie! "hot" needs to look back at "drinks".

### The fix: attention

**Attention = let each word look at the other words, and listen more to the useful ones.**

*Everyday example:* in a noisy room you hear many voices, but you listen **more** to the person talking to you.

3 small steps:

1. **Score:** dot product of "hot" with each word in the sentence. A big score means "useful".
2. **Softmax:** scores → weights that add up to 1.
3. **Weighted average:** multiply each word's vector by its weight, then add. The result is a new "hot" that **carries the clue from "drinks"**.

> **This is the heart of the Transformer.** The picture at the top: the purple *Attention* box does exactly these 3 steps.

In [ ]:
query = vectors[-1]                  # the last word: "hot"

att_scores = vectors @ query         # 1. hot · ali, hot · drinks, hot · hot
print("scores: ", att_scores)        # [0, 2, 3]

weights = softmax(att_scores)        # 2. softmax
print("weights:", weights)           # [0.035, 0.259, 0.705]

mix = weights @ vectors              # 3. weighted average of the 3 vectors
print("hot before:", query)          # [0, 1, 1, 1]
print("hot after: ", mix)            # [0.105, 1.224, 0.705, 0.705]: drink went up!

Look at "hot": the **drink** number went **up** (1 → 1.22) and the **food** number went **down** (1 → 0.71). The word "hot" now knows it is about a **drink**.

**✏️ Quick Check:** Do the weights add up to 1? Print `weights.sum()`. Which word does "hot" listen to the least?

In [ ]:
# your code here

**Answer:**

In [ ]:
print(round(weights.sum(), 3))     # 1.0 (the computer may show 0.9999999: tiny rounding)
# "ali": only 3.5%. A name tells us nothing about chai or roti.

## Layer 5: Score every word, softmax, pick

We have a better "hot". Now compare it with **every word in the vocabulary** (dot product again), turn the scores into chances (softmax), and pick the biggest.

In [ ]:
scores = E @ mix                     # dot product of the mixed "hot" with every word
chances = softmax(scores)

for i in np.argsort(-chances):       # biggest chance first
    print(f"{vocab[i]:7} score {scores[i]:.2f}   chance {chances[i]:.1%}")

print("Next token:", vocab[chances.argmax()])    # chai

**The tie is gone: chai wins (37%), roti only 13%.** Attention fixed what counting could not.

**Why is "hot" in 2nd place?** Our toy is tiny and uses the same vectors to read and to guess. A real model has a separate, learned output layer and has read enough text to know that "hot hot" is rare.

**✏️ Quick Check:** Change the sentence to **"ali eats hot"** and run Layers 4–5 again. Which word wins? *(Try it before the Home Problem!)*

In [ ]:
# your code here

**Answer:**

In [ ]:
v = E[[vocab.index(t) for t in "ali eats hot".split()]]
m = softmax(v @ v[-1]) @ v
c = softmax(E @ m)
print(vocab[c.argmax()], f"{c.max():.1%}")    # roti 37.4%

## Layer 6: Put it together, and repeat

One function = Layers 3 to 5. Then we **add the new token and run again**. This is how an LLM writes a whole answer, one word at a time.

In [ ]:
def next_token(words):
    vectors = E[[vocab.index(w) for w in words]]      # tokens → IDs → vectors
    weights = softmax(vectors @ vectors[-1])          # attention weights for the last word
    mix = weights @ vectors                           # weighted average
    chances = softmax(E @ mix)                        # score every word → chances
    return vocab[chances.argmax()], chances


words = ["ali", "drinks", "hot"]
for step in range(3):                                 # 3 passes = 3 new tokens
    word, chances = next_token(words)
    print(" ".join(words), "→", word, f"({chances.max():.0%})")
    words.append(word)                                # add the guess, then repeat

**✏️ Quick Check:** The toy says "chai chai chai". Why would a real LLM not do this?

In [ ]:
# your code here

**Answer:** Our toy has only 6 words and was never trained. A real LLM learned from huge amounts of text that "chai chai chai" almost never happens, so it gives other tokens a bigger chance. (Real chatbots also sometimes pick a token that is not the top one.)

In [ ]:
# no code needed

## Layer 7: The real LLM

Our toy and a real LLM follow **the same recipe**. Look at the picture again:

![Transformer, annotated](Visual%20Explanation/Day_23_Transformer_Annotated.png)

| In our toy | In a real LLM |
|---|---|
| 6 words | about 100,000 tokens |
| 4 numbers per word, **we wrote them** | 768 to 12,000+ numbers per word, **learned** |
| **1** attention step | attention + a *feed-forward* step, **repeated 12 to 100+ times** (the purple block) |
| nothing learned | trained on huge text (see below) |

**Feed-forward** = a small network (like your Day 19 Dense layer) that "thinks" about each word after attention has mixed the words.

### How is it trained?

It is the **same loop as Day 19–22**:

1. **Pre-training:** show a sentence with the last word hidden (`ali drinks hot ___`). The model guesses. The **loss** (cross entropy, Day 22) says how wrong it was. **Gradient descent** changes the weights a little. Repeat billions of times on books, websites and code.
2. **Fine-tuning:** train more on good question-and-answer examples (and human ratings), so it behaves like a helpful assistant.
3. **Use (inference):** the weights are now fixed. It only does the steps we built today: guess, add, repeat.

In [ ]:
print("Our toy has", E.size, "learned numbers")   # 6 words × 4 numbers = 24
# Day 19 network: 147   |   Day 20 network: 5,130   |   a big LLM: 100,000,000,000+

**✏️ Quick Check:** A real model has 100,000 tokens and 768 numbers per token. How many numbers are in its embedding table?

In [ ]:
# your code here

**Answer:** and this is only the embedding table. The attention and feed-forward blocks add many more.

In [ ]:
print(100_000 * 768)    # 76,800,000: 76.8 million, and that is only the first table

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [ ]:
import numpy as np

def softmax(scores):
    e = np.exp(scores)
    return e / e.sum()

# Layer 3: tokens → IDs → embeddings (person, drink, food, warm)
vocab = ["ali", "drinks", "eats", "hot", "chai", "roti"]
tokens = "ali drinks hot".split()
ids = [vocab.index(t) for t in tokens]
E = np.array([[3, 0, 0, 0], [0, 2, 0, 0], [0, 0, 2, 0],
              [0, 1, 1, 1], [0, 2, 0, 1], [0, 0, 2, 1]], dtype=float)
vectors = E[ids]

# Layer 4: attention for the last word
weights = softmax(vectors @ vectors[-1])
mix = weights @ vectors

# Layer 5: score every word, softmax, pick
chances = softmax(E @ mix)
print("Next token:", vocab[chances.argmax()], f"({chances.max():.1%})")   # chai (37.4%)

### Key takeaways

- An **LLM** = **L**arge **L**anguage **M**odel: a very big autocomplete. One job: guess the next token, then repeat.
- **Counting** (Layers 1–2) breaks: ties, and it only knows phrases it has seen.
- **Words → numbers** (tokens, IDs, embeddings): similar words get similar numbers.
- **Attention** fixes the tie: each word listens more to the useful words. It is the heart of the **Transformer**.
- **Softmax** turns scores into chances (same as Day 21). Training uses **cross entropy** (same as Day 22).
- A real LLM = the same block **stacked many times**, with billions of learned numbers. An **SLM** is the same idea, just smaller.

---
# 🏠 Home Problem: "ali eats hot ___"

Do the **same layers** for the new sentence.

1. Tokens and IDs (Layer 3).
2. Vectors (`E[ids]`).
3. Attention: scores, weights, mix (Layer 4). Compare "hot" before and after: which number went up?
4. Score every word, softmax (Layer 5). What is the next token, and its chance?
5. In 2 lines: only one word changed. Which layer made the answer change, and how?